In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/)
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
import lightgbm as lgb
from sentence_transformers import CrossEncoder, InputExample
import wandb

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# MILESTONE 2

# Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [6]:
from datasets import load_dataset
# Load dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# Map function to concatenate prompt and A with a space
def concat_prompt_a(example):
    example['combined_text'] = f"{example['prompt']} {example['A']}"
    return example
dataset = dataset.map(concat_prompt_a)
row_51 = dataset['train'][51]
combined = row_51['combined_text']
print("--- ROW 51 COMBINED TEXT ---")
print(combined)
print("--- CHARACTER LENGTH ---")
print(len(combined))

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

--- ROW 51 COMBINED TEXT ---
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.
--- CHARACTER LENGTH ---
614


# Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.vocab_size)

30522


# Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token. 

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print(tokenizer.sep_token_id)

102




# Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). What is the exact geometric shape (dimensions) of the resulting input_ids tensor?
****

In [13]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load train dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
prompts = list(dataset['train']['prompt'])

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize
encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print(encoded['input_ids'].shape)

torch.Size([2000, 128])


# BERT/RoBERTa Architecture & Attention Mechanisms

# A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

# In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [14]:
hidden_size = 768
num_attention_heads = 12

# Dimensionality of each individual attention head
head_dim = hidden_size // num_attention_heads
print(head_dim)

64



# Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. What is the exact shape of the last_hidden_state tensor returned? Note: We follow zero-indexing here.


In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load train dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
prompt_0 = dataset['train'][0]['prompt']
print("Prompt at index 0:", prompt_0)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

# Tokenize
inputs = tokenizer(prompt_0, return_tensors='pt')
print("Inputs keys:", inputs.keys())
print("Tokens length:", inputs['input_ids'].shape)

# Forward pass
outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state
print("last_hidden_state shape:", last_hidden_state.shape)

Prompt at index 0: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Inputs keys: KeysView({'input_ids': tensor([[  101,  4060,  1996,  2190,  2825,  3437,  1024,  2054,  2003,  3235,
          2002,  5178, 13327,  1005,  1055,  3193,  2006,  1996,  3276,  2090,
          2051,  1998,  2529,  4598,  1029,  2426,  1996,  3205,  7047,  1012,
           102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]])})
Tokens length: torch.Size([1, 31])
last_hidden_state shape: torch.Size([1, 31, 768])


# Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [16]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# Load the dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
prompt_0 = dataset['train'][0]['prompt']

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased')

# Tokenize
inputs = tokenizer(prompt_0, return_tensors='pt')

# Pass through the model
outputs = model(**inputs)
last_hidden_state = outputs.last_hidden_state

# Extract CLS token vector (index 0 of batch, index 0 of sequence)
cls_vector = last_hidden_state[0, 0]

# Sum first 5 values
first_5_sum = sum(cls_vector[:5].tolist())

print("First 5 values:", cls_vector[:5].tolist())
print("Sum of first 5 values:", first_5_sum)
print("Rounded sum:", round(first_5_sum, 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


First 5 values: [-0.46766412258148193, -0.07544441521167755, -0.20190182328224182, -0.007064236328005791, -0.4480222463607788]
Sum of first 5 values: -1.200096843764186
Rounded sum: -1.2001


# Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

# What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [17]:
from transformers import AutoTokenizer, AutoModel

text = "Light-ion fusion is a technique."

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)

# Tokenize
inputs = tokenizer(text, return_tensors='pt')
input_ids = inputs['input_ids'][0]
tokens = tokenizer.convert_ids_to_tokens(input_ids)

# Find token index of "fusion"
fusion_idx = tokens.index("fusion")  # Index 4

# Pass through the model
outputs = model(**inputs)

# Last layer (index -1), first head (index 0)
attention_matrix = outputs.attentions[-1][0, 0]

# Attention from [CLS] (index 0) to "fusion" (index 4)
attention_weight = attention_matrix[0, fusion_idx].item()

print(f"Fusion token index: {fusion_idx}")
print(f"Raw weight: {attention_weight}")
print(f"Rounded weight: {round(attention_weight, 4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fusion token index: 4
Raw weight: 0.10247313976287842
Rounded weight: 0.1025


# Context-Aware Embeddings 
# Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [19]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

# Load dataset
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row_0 = dataset['train'][0]
prompt = row_0['prompt']
option_b = row_0['B']

# Initialize the model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Generate embeddings
emb_prompt = model.encode(prompt, convert_to_tensor=True)
emb_option_b = model.encode(option_b, convert_to_tensor=True)

# Calculate similarity
similarity = util.cos_sim(emb_prompt, emb_option_b).item()
print(round(similarity, 4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


0.7658


# Build two complete ranking pipelines evaluating every row in train.csv.

# Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

# Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

# First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

# Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [21]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Define MAP@3 inline
def map3(predictions, targets):
    scores = []
    for pred, target in zip(predictions, targets):
        score = 0.0
        for i, p in enumerate(pred[:3]):
            if p == target:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

# Load data (specify exact absolute paths if needed)
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
targets = train_df['answer'].tolist()
options = ['A', 'B', 'C', 'D', 'E']

# --- TF-IDF Cosine Similarity ---
all_texts = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_texts.extend(train_df[col].astype(str).tolist())
    all_texts.extend(test_df[col].astype(str).tolist())

tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_vectorizer.fit(all_texts)

num_samples = len(train_df)
tfidf_scores = np.zeros((num_samples, 5))
train_prompt_vecs = tfidf_vectorizer.transform(train_df['prompt'].astype(str))

for idx, col in enumerate(options):
    train_option_vecs = tfidf_vectorizer.transform(train_df[col].astype(str))
    for i in range(num_samples):
        p_vec = train_prompt_vecs[i]
        o_vec = train_option_vecs[i]
        tfidf_scores[i, idx] = cosine_similarity(p_vec, o_vec)[0, 0]

tfidf_preds = []
for i in range(num_samples):
    sorted_idx = np.argsort(tfidf_scores[i])[::-1]
    tfidf_preds.append([options[idx] for idx in sorted_idx[:3]])

# --- Sentence-Transformers Cosine Similarity ---
sbert_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompt_emb = sbert_model.encode(train_df['prompt'].astype(str).tolist(), batch_size=128, show_progress_bar=False)
prompt_emb_norm = prompt_emb / np.linalg.norm(prompt_emb, axis=1, keepdims=True)

sbert_scores = np.zeros((num_samples, 5))
for idx, col in enumerate(options):
    opt_emb = sbert_model.encode(train_df[col].astype(str).tolist(), batch_size=128, show_progress_bar=False)
    opt_emb_norm = opt_emb / np.linalg.norm(opt_emb, axis=1, keepdims=True)
    sims = np.sum(prompt_emb_norm * opt_emb_norm, axis=1)
    sbert_scores[:, idx] = sims

sbert_preds = []
for i in range(num_samples):
    sorted_idx = np.argsort(sbert_scores[i])[::-1]
    sbert_preds.append([options[idx] for idx in sorted_idx[:3]])

# --- Evaluation ---
sbert_map3 = map3(sbert_preds, targets)
print(f"MiniLM MAP@3: {sbert_map3:.6f}")

count = 0
for i in range(num_samples):
    target = targets[i]
    in_tfidf = target in tfidf_preds[i]
    in_sbert = target in sbert_preds[i]
    if (not in_tfidf) and in_sbert:
        count += 1

print("Count:", count)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3: 0.423083
Count: 584


# Zero-shot classification concepts 
# Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [24]:
import pandas as pd
from transformers import pipeline

# Load data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row_1 = train_df.iloc[1]
prompt = row_1['prompt']
options = [row_1['A'], row_1['B'], row_1['C']]

# Initialize zero-shot pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Run classifier
result = classifier(prompt, candidate_labels=options)

# Top-ranked option score
top_score = result['scores'][0]
print(round(top_score, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

0.4575


# Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

# What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [26]:
import pandas as pd
from transformers import pipeline

# Load data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row_1 = train_df.iloc[1]
prompt = row_1['prompt']
options = [row_1['A'], row_1['B'], row_1['C']]

# Initialize zero-shot pipeline
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Softmax (multi_label=False)
res_softmax = classifier(prompt, candidate_labels=options, multi_label=False)
sum_softmax = sum(res_softmax['scores'])

# Sigmoid (multi_label=True)
res_sigmoid = classifier(prompt, candidate_labels=options, multi_label=True)
sum_sigmoid = sum(res_sigmoid['scores'])

# Absolute difference
abs_diff = abs(sum_softmax - sum_sigmoid)
print(round(abs_diff, 4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

0.9995


# Let's try Generative AI instead of Classification. 

# Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
# Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 


In [29]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load data and construct prompt for row 0
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
row_0 = train_df.iloc[0]
prompt = row_0['prompt']
A = row_0['A']
B = row_0['B']

input_text = f"Question: {prompt}. Is the correct answer A: {A} or B: {B}? Answer with just the letter A or B."

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# Tokenize and generate
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=5)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(repr(generated_text))


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


'B'
